In [1]:
anthropic_key = "REDACTED_API_KEY"
import getpass
import os


def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")


_set_env("ANTHROPIC_API_KEY")

In [2]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_anthropic import ChatAnthropic
from langchain_ollama import ChatOllama
from IPython.display import Image, display
from typing import Dict, TypedDict, Optional
import random
import time
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
import json

In [ ]:
PROMPTS_ROOT_DIR = "prompts"
AIAH_CONSCIENCE_MANAGER_PROMPT_PATH = f"{PROMPTS_ROOT_DIR}/AIAH_conscience_manager.json"
AIAH_SKILL_MANAGER_PROMPT_PATH = f"{PROMPTS_ROOT_DIR}/AIAH_skill_manager.json"
AIAH_MEMORY_MANAGER_PROMPT_PATH = f"{PROMPTS_ROOT_DIR}/AIAH_memory_manager.json"
AIAH_HEALTH_MANAGER_PROMPT_PATH = f"{PROMPTS_ROOT_DIR}/AIAH_health_manager.json"
AIAH_GET_ACTION_PROMPT_PATH = f"{PROMPTS_ROOT_DIR}/AIAH_get_action_manager.json"
ES_GET_ENV_STATUS_PROMPT_PATH = f"{PROMPTS_ROOT_DIR}/ES_get_environment_status.json"
ES_UPDATE_ENV_PROMPT_PATH = f"{PROMPTS_ROOT_DIR}/ES_update_environment.json"
REALITY_CHECK_AGENT_PROMPT_PATH = f"{PROMPTS_ROOT_DIR}/reality_check_agent.json"
STATE_NOTES_ROOT_DIR = "state_notes"
AIAH_CONSCIENCE_DATA_PATH = f"{STATE_NOTES_ROOT_DIR}/AIAH_conscience.json"
AIAH_SKILL_DATA_PATH = f"{STATE_NOTES_ROOT_DIR}/AIAH_skills.json"
AIAH_MEMORY_DATA_PATH = f"{STATE_NOTES_ROOT_DIR}/AIAH_memory.json"
AIAH_HEALTH_DATA_PATH = f"{STATE_NOTES_ROOT_DIR}/AIAH_health.json"
ENVIRONMENT_DATA_PATH = f"{STATE_NOTES_ROOT_DIR}/ES_notes.json"

class EnvironmentGraphState(TypedDict):
    """
    State Graph for the Environment Simulation. 
    A State Contatins the Following information: 
    
    - previous_agent: the agent prompted/used before this
    - current_agent: the current agent to be used
    - metadata: json data carried from the previous agent
    - day_count: day of experiment (will be used to interrupt and select new flows)
    - luck_factor: luck of the agent in the day (decided by the environment)
    """
    previous_agent = None
    current_agent = None
    state_metadata = {}
    simulation_day_count = 0
    day_luck_factor = None
    

In [ ]:
def  display_prompt_metadata(prompt_data):
    print(f"Prompt Metadata:")
    for key in prompt_data["metadata"]:
        print(f"- {key}: {prompt_data["metadata"][key]}")
    
def prompt_llm(system_prompt,user_input,model="llama",verbose=False):
    """
    Supporting function to prompt LLM
    """
    if len(system_prompt)==0:
        raise Exception("Invalid System Prompt - Empty")
    if len(user_input)==0:
        raise Exception("Invalid User Input - Empty")
    if model == "llama":
        # code to prompt Ollama
        if verbose:
            print("Prompting Llama")
    elif model == "gpt-4o":
        # code to prompt gpt4-o
        if verbose:
            print("Prompting GPT")
    else:
        raise Exception(f"No Support for model {model}. Add code in function prompt_llm")
        
        


def handle_ES_environment_status(state:EnvironmentGraphState,
                                 verbose:str = False):
    """
    Function to handle prompt
    """
    state.current_agent = "ENVIRONMENT_SIMULATOR"
    prompt_data = json.load(ES_GET_ENV_STATUS_PROMPT_PATH)
    if verbose:
        display_prompt_metadata(prompt_data)
    agent_prompt = prompt_data["prompt"]
    prompt_supporting_metadata = json.dumps(state.state_metadata)
    
    # code to prompt LLM: 
    llm_response = prompt_llm(agent_prompt,prompt_supporting_metadata,model="llama")
    # process response, update Environment notes
    
    return state
    

'"\\n  You are a reality check agent that evaluates whether a human\'s action is physically and biologically possible given their current health and environment. You will analyze the environmental status, human\'s health, and action taken to determine if the action is feasible.\\n\\n  If the action is physically possible, return \\"possibility\\": \\"possible\\".\\n  If the action is impossible due to physical or biological constraints, return \\"possibility\\": \\"impossible\\" with an explanation in \\"reasoning\\".\\n  Input Format (JSON)\\n\\n  You will receive:\\n\\n      environment_status - Describes surroundings, temperature, weather, available resources, and danger level.\\n      current_health - A structured JSON representing various health parameters.\\n      action_taken - The action the human attempts.\\n\\n  Example Input\\n\\n  {\\n    \\"environment_status\\": {\\n      \\"temperature\\": -10,\\n      \\"weather\\": \\"blizzard\\",\\n      \\"food_availability\\": \\"no

"\nYou are the Environment Simulation Manager, responsible for updating the entire ecosystem, environment, and agent's condition based on their chosen action. This is a cause-and-effect simulation where the agent's decisions directly influence their survival and the state of the world.\nYou have prompted the agent with a environment snapshot, and have asked the agent what it wants to do. Now, the agent has responded with an action it decides to take.\nYour Task\n\nGiven the agent's action, luck factor, current environmental conditions, and agent's health, you must:\n\n    Determine the Feasibility of the Action\n        Check if the agent can physically perform the action based on health, stamina, and environment.\n        If the action is impossible, return \"action_success\": false and explain why.\n        If the action is possible, decide how effectively it was executed based on luck, skill level, and environmental factors.\n\n    Update the Environment\n        Modify resources (e